# 类型转换：让数据"变形"成需要的样子

---

**学习目标**

1. 掌握 8 个转换函数：`int` `float` `str` `bool` `list` `tuple` `dict` `set`
2. 分清**隐式转换**（Python 自动做）和**显式转换**（你手动做）
3. 记住"假值清单"—— 这是 `if` 判断的基础
4. 知道哪些转换一定会失败

**一句话理解**

类型转换就是"换包装"。同一份数据，换成合适的包装才能参与对应的运算：
`'1' + '2'` 是拼接，`int('1') + int('2')` 才是加法。


---
## 一、隐式转换 vs 显式转换

**隐式转换**：Python 自动帮你转，你什么都不用写。

```python
1 + 2.5    ->  3.5      （int 自动变成 float）
True + 1   ->  2        （bool 自动变成 int）
```

规则：**精度低的向精度高的看齐**。`bool < int < float < complex`

**显式转换**：你调用转换函数，自己指定目标类型。下面正式开始。


In [ ]:
print('int + float =', 1 + 2.5, type(1 + 2.5))
print('bool + int  =', True + 1, type(True + 1))
print('int + complex =', 1 + (2 + 3j), type(1 + (2 + 3j)))


---
## 二、转成整数 `int()`

规则：**直接截断小数部分，不四舍五入**。


In [ ]:
print(int(3.14))        # 3
print(int(3.99))        # 3   —— 注意！不是 4
print(int(-3.99))       # -3  —— 是"向零取整"，不是向下取整
print(int('10'))        # 10
print(int('  42  '))    # 42  —— 会自动去掉首尾空白
print(int(True))        # 1
print(int(False))       # 0
print(int())            # 0   —— 不传参数默认给 0

# 【对比】三种"取整"的区别，容易混：
import math
print('int(-3.99)   =', int(-3.99))        # -3  向零取整（截断）
print('math.floor(-3.99) =', math.floor(-3.99))   # -4  向下取整
print('math.ceil(-3.99)  =', math.ceil(-3.99))    # -3  向上取整
print('round(-3.99) =', round(-3.99))      # -4  四舍五入
print('7 // 2 =', 7 // 2, '  -7 // 2 =', -7 // 2)  # 整除也是向下取整


### 转不了的情况：字符串里有非数字内容


In [ ]:
for s in ['10', '3.14', 'abc', '', '10个']:
    try:
        print(f'int({s!r}) = {int(s)}')
    except ValueError as e:
        print(f'int({s!r}) 失败 -> {e}')

# **关键结论**：
# - `int('3.14')` **会失败**！因为小数点不是整数字符。要先用 `float()` 再用 `int()`
print('两步走：', int(float('3.14')))     # 3


---
## 三、转成浮点数 `float()`


In [ ]:
print(float(5))          # 5.0
print(float('3.14'))     # 3.14
print(float('1e3'))      # 1000.0  —— 支持科学计数法
print(float(True))       # 1.0
print(float('inf'))      # inf     —— 无穷大
print(float('nan'))      # nan     —— Not a Number

# 坑：float('abc') 会报错
try:
    float('abc')
except ValueError as e:
    print('float("abc") 失败 ->', e)

# 实用：判断一个 nan
import math
n = float('nan')
print('nan == nan ?', n == n)              # False！nan 不等于任何东西，包括自己
print('正确的判断方式：', math.isnan(n))     # True


---
## 四、转成字符串 `str()`

这个几乎不会失败 —— **任何对象都能转成字符串**。


In [ ]:
print(repr(str(100)))       # '100'
print(repr(str(2.718)))     # '2.718'
print(repr(str(True)))      # 'True'
print(repr(str([1, 2])))    # '[1, 2]'
print(repr(str(None)))      # 'None'

class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

p = Point(1, 2)
print('自定义对象也能转：', str(p))       # 默认输出类似 <__main__.Point object at 0x...>
# 想让它输出得好看，可以自己定义 __str__ 方法（面向对象那章会讲）


**实用场景**：拼 SQL、拼文件名、拼 URL 时，数字必须转成字符串。


In [ ]:
user_id = 42
filename = 'user_' + str(user_id) + '.txt'
print(filename)
print(f'或者更简单：user_{user_id}.txt')


---
## 五、转成布尔值 `bool()` 与"假值清单"

这是**最重要的一节**。因为 `if x:` 本质就是 `if bool(x):`

### 一共只有 8 类东西会被判为 `False`：

| 类别 | 例子 |
|---|---|
| 布尔假 | `False` |
| 整数零 | `0` |
| 浮点零 | `0.0` |
| 复数零 | `0j` |
| 空字符串 | `''` |
| 空列表 | `[]` |
| 空元组 | `()` |
| 空字典 / 空集合 | `{}` / `set()` |
| 空值 | `None` |

**除此之外，一切皆为 `True`。**


In [ ]:
falsy_values = [False, 0, 0.0, 0j, '', [], (), {}, set(), None]
for v in falsy_values:
    print(f'{repr(v):>10} -> {bool(v)}')

print('\n--- 一些容易判错的例子 ---')
print("bool('0')      ->", bool('0'), '  （非空字符串！是 True）')
print("bool('False')  ->", bool('False'), '  （非空字符串！也是 True）')
print('bool([0])      ->', bool([0]), '  （非空列表！是 True）')
print('bool(-1)       ->', bool(-1), '  （非零就是 True）')


### 这张表能帮你写出很简洁的代码


In [ ]:
cart = []
# 啰嗦写法
if len(cart) != 0:
    print('购物车有东西')

# 地道写法：空列表本身就是"假"
if cart:
    print('购物车有东西')

# 处理"可能为空"的值
def get_name(user):
    return user.get('name') if user else None

user = {'name': ''}
name = get_name(user)
print('显示名：', name or '未填写')     # 空字符串是假值，所以走 or 右边


---
## 六、容器之间的转换

### 6.1 转列表 `list()`


In [ ]:
print(list('hello'))          # ['h', 'e', 'l', 'l', 'o']  字符串按字符拆
print(list((1, 2, 3)))        # [1, 2, 3]
print(list({1, 2, 3}))        # 集合转列表（顺序不保证）
print(list({'a': 1, 'b': 2})) # ['a', 'b']  字典只取【键】
print(list(range(3)))         # [0, 1, 2]


### 6.2 转元组 `tuple()`

用途：把列表变成**不可变**的版本，防止被意外修改，也可以当字典的键。


In [ ]:
print(tuple([1, 2, 3]))       # (1, 2, 3)
print(tuple('world'))         # ('w', 'o', 'r', 'l', 'd')

# 元组可以当字典的键，列表不行 —— 这是上一课"可变/不可变"的直接应用
location = {(31.2, 121.5): '上海', (39.9, 116.4): '北京'}
print('上海在：', location[(31.2, 121.5)])

try:
    bad = {[1, 2]: 'x'}
except TypeError as e:
    print('用列表当键 -> ', e)


### 6.3 转集合 `set()`：去重神器


In [ ]:
nums = [1, 2, 2, 3, 3, 3, 4]
print('去重后：', set(nums))                       # {1, 2, 3, 4}
print('长度变化：', len(nums), '->', len(set(nums)))

# 顺手做集合运算（交、并、差）
a, b = {1, 2, 3}, {2, 3, 4}
print('交集：', a & b)
print('并集：', a | b)
print('差集：', a - b)

# 实用：找出两个列表里重复的元素
list1 = ['张三', '李四', '王五']
list2 = ['李四', '王五', '赵六']
print('两人都选了的：', set(list1) & set(list2))


### 6.4 转字典 `dict()`

输入必须是"一堆键值对"，通常是**二元组的列表**。


In [ ]:
print(dict([('name', 'Alice'), ('age', 25)]))     # {'name': 'Alice', 'age': 25}
print(dict((['x', 1], ['y', 2])))                 # {'x': 1, 'y': 2}
print(dict(zip(['a', 'b'], [1, 2])))              # {'a': 1, 'b': 2}  —— zip 配对后转字典，超常用
print(dict(name='Bob', age=30))                   # 关键字参数形式

# 两个列表转字典的完整场景
keys = ['语文', '数学', '英语']
values = [90, 85, 78]
score_dict = dict(zip(keys, values))
print('成绩单：', score_dict)


---
## 七、字符与编码：`chr()` 和 `ord()`

这两个函数是做**字符编码转换**的，理解它们就理解了"字符本质上是数字"。


In [ ]:
print('chr(65)  ->', chr(65))      # A    数字 -> 字符
print('ord("A") ->', ord('A'))     # 65   字符 -> 数字
print('chr(97)  ->', chr(97))      # a
print('ord("中") ->', ord('中'))   # 20013 Unicode 码点

# 实用 1：生成字母表（不用手打 26 个字母）
alphabet = [chr(i) for i in range(ord('a'), ord('z') + 1)]
print('字母表：', ''.join(alphabet))

# 实用 2：字母大小写转换的底层原理（差值是固定的 32）
print('a -> A 需要减：', ord('a') - ord('A'))
print('小写转大写：', chr(ord('h') - 32))

# 实用 3：凯撒密码（每个字母往后移 3 位）——理解"字符即数字"的最佳例子
def caesar(text, shift=3):
    result = []
    for ch in text:
        if ch.isalpha():
            base = ord('a') if ch.islower() else ord('A')
            result.append(chr((ord(ch) - base + shift) % 26 + base))
        else:
            result.append(ch)
    return ''.join(result)

msg = 'hello world'
encrypted = caesar(msg)
print(f'原文：{msg}')
print(f'加密：{encrypted}')
print(f'解密：{caesar(encrypted, shift=-3)}')


---
## 八、复杂数 `complex()`


In [ ]:
print(complex(3.14))         # (3.14+0j)
print(complex('2+3j'))       # (2+3j)
print(complex(2, 3))         # (2+3j)

z = 3 + 4j
print('实部：', z.real, ' 虚部：', z.imag, ' 模：', abs(z))


---
## 九、一张总表：能转吗？

| 目标 \ 来源 | `int` | `float` | `str` | `bool` |
|---|---|---|---|---|
| → `int` | ✓ | ✓（截断） | 仅纯数字串 | ✓ |
| → `float` | ✓ | ✓ | 数字串 / `inf` / `nan` | ✓ |
| → `str` | ✓ | ✓ | ✓ | ✓（近乎万能） |
| → `bool` | ✓ | ✓ | ✓ | ✓（看假值清单） |

**一定会失败的三类**：

1. `int('3.14')` —— 有小数点，要先 `float()`
2. `int('abc')` / `float('abc')` —— 非数字内容
3. `int('')` —— 空字符串（但 `bool('')` 是 `False`）


---
## 十、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `int('3.14')` | `ValueError` | `int(float('3.14'))` |
| 以为 `int()` 会四舍五入 | `int(3.99)` 得 3 | 要四舍五入用 `round()` |
| `float('nan') == float('nan')` | `False` | 用 `math.isnan()` |
| `bool('0')` / `bool('False')` | 都是 `True` | 非空字符串一律为真 |
| `dict([1, 2, 3])` | `TypeError` | 必须传"键值对"序列 |
| `list(字典)` 想拿值 | 只拿到键 | 用 `list(d.values())` |
| `set()` 丢掉了顺序 | 输出顺序是乱的 | 需要顺序就排序：`sorted(set(...))` |
| 转字符串忘了 `str()` | `TypeError: 只能拼接 str` | `str(x)` 或 f-string |
| 用 `int()` 处理用户输入不校验 | 输入非数字直接崩 | `try/except` 包起来 |


---
## 十一、动手练习

**练习 1**：`'  123  '` 能转成 `int` 吗？`'12.0'` 能吗？`'1,234'` 能吗？逐个验证。

**练习 2**：用户输入 `"3.14"`（字符串），怎么安全地拿到它的整数部分？

**练习 3**：把 `[('a', 1), ('b', 2)]` 转成字典，再转回"两个列表"。

**练习 4**：给定一个成绩列表 `[88, 45, 92, 61, 77, 92]`，用集合去重后排序输出。

**练习 5**：判断一个字符串是不是"纯数字"，不要用 `isdigit`，用 `try/except int()` 实现。

**练习 6**：用 `chr` 和 `ord` 判断一个字符是不是大写字母（不用 `.isupper()`）。

**练习 7**：下面代码为什么不进入 if 分支？改对。
```python
score = input('输入分数：')
if score > 60:
    print('及格')
```

---

**答案思路（先自己做）**

```python
# 练习 1
print(int('  123  '))     # 123，会去掉首尾空白
# int('12.0')  -> ValueError
# int('1,234') -> ValueError（逗号不是数字字符）

# 练习 2
s = '3.14'
print(int(float(s)))      # 3

# 练习 3
d = dict([('a', 1), ('b', 2)])
print(d)
print(list(d.keys()), list(d.values()))

# 练习 5
def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

# 练习 7
score = int(input('输入分数：'))
if score > 60:
    print('及格')
```
